[Guide Index](README.md)


# 05d The SHACL Library Registry

`starlayer.starontology` bundles six ontology/SHACL-shapes pairs, each describing a vocabulary this stack uses internally:

| Prefix | Pair | What it is |
|---|---|---|
| `manch:` | `manchester_owl` / `manchester_shacl` | OWL 2 Manchester Syntax's own internal AST as RDF |
| `skos:` | `skos_owl` / `skos_shacl` | The W3C's own SKOS vocabulary (`skos.rdf`, converted to Turtle) and StarLayer's shapes for the Reference's numbered conditions |
| `salg:` | `sparql_owl` / `sparql_shacl` | A SPARQL 1.2 query/update's own algebra as RDF (internal: `get_ontology_list()` leaves it out, `get_ontology_graph()` still loads it) |
| `sqe:` | `sqe_owl` / `sqe_shacl` | A SPARQL query's raw parse tree (pre-`translateQuery`), for hand/UI editing |
| `spexpr:` | `spexpr_owl` / `spexpr_shacl` | The expression, variable and property-path vocabulary the SPARQL trees share |
| `srl:` | `srl_owl` / `srl_shacl` | SRL's own rule-set abstract syntax as RDF |

Plus a thirteenth entry, `shacl_shacl` — the SHACL 1.2 meta-shapes themselves (shapes-about-shapes). It is built from pySHACL's own shacl-shacl and StarLayer's validation shapes by `scripts/build.py`, and stored as `shacl12-shacl.ttl`, so it has no ontology counterpart.

Every entry is one Turtle file in `starlayer/starontology/`. The registry parses only the file you ask for, once per process, and hands you a fresh copy each time. `starontology.trig` holds the same graphs as named graphs in a single file, for tools that want them all at once; it is generated, and the library never reads it. Edit the `.ttl` files (never `shacl12-shacl.ttl`, `skos-ontology.ttl` or the `.trig`, which `python scripts/build.py` regenerates). Each entry also has a real, fetchable URL (section 6 below).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.


In [1]:
from starlayer import starontology

## 1. List what's registered

`get_ontology_list()` returns every entry's metadata (`name`, `description`, `type`), sorted by name — no `pyshacl` import needed for this, just reading the registry.

In [2]:
for info in starontology.get_ontology_list():
    print(f"{info.name:20} {info.type:10} {info.description[:80]}")

manchester_owl       syntax-tree-graph OWL 2 Manchester Syntax's own internal syntax tree, as RDF (manch:) - frames/cla
manchester_shacl     shacl      SHACL shapes validating a manch: syntax tree's own structural well-formedness.
shacl_shacl          shacl      The SHACL 1.2 meta-shapes themselves (starlayer.shacl's own shapes-about-shapes,
skos_owl             owl        The W3C's own SKOS vocabulary (skos.rdf from the SKOS Reference, W3C Recommendat
skos_shacl           shacl      SHACL shapes for SKOS: the Reference's numbered conditions as validation shapes,
spexpr_owl           syntax-tree-graph The SPARQL expression/variable/property-path vocabulary (spexpr:), carved out of
spexpr_shacl         shacl      SHACL shapes validating an spexpr: expression/variable/path value's own well-for
sqe_owl              syntax-tree-graph A SPARQL SELECT/ASK/CONSTRUCT query's own raw parse tree, pre-translateQuery, as
sqe_shacl            shacl      SHACL shapes validating an sqe: query-edit gr

## 2. Load an entry's graph

`get_ontology_graph(name)` returns a fresh `StarLayerGraph` of the named file — a real, standalone artifact usable by any RDF tool, not something the registry itself invents. Raises `KeyError` (naming every valid choice) for a typo'd name, rather than returning `None`.

In [3]:
skos_owl = starontology.get_ontology_graph("skos_owl")
skos_shacl = starontology.get_ontology_graph("skos_shacl")
print("skos_owl:  ", len(skos_owl), "triples")
print("skos_shacl:", len(skos_shacl), "triples")

skos_owl:   252 triples
skos_shacl: 302 triples


## 3. Validate real data against a registered shapes graph

Any `*_shacl` entry works directly with `StarShaclSchema`, the same as a hand-authored shapes graph.

In [4]:
from starlayer import StarLayerGraph, StarShaclSchema

data = StarLayerGraph()
data.parse(data="""
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
    @prefix ex: <http://example.org/> .
    ex:scheme1 a skos:ConceptScheme .
    ex:concept1 a skos:Concept ; skos:inScheme ex:scheme1 ; skos:prefLabel "Example"@en .
""", format="turtle")

result = StarShaclSchema(shacl_graph=skos_shacl).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)

conforms: True


## 4. Every shapes graph belongs to a StarLayer profile

StarLayer describes its own conformance in one document, `starlayer-profiles.ttl`. It defines a *StarLayer profile* for each vocabulary (`slp:Skos`, `slp:Manchester`, `slp:SparqlQuery`, `slp:SparqlRL`, `slp:Shacl`). Each is a `prof:Profile` that is `prof:isProfileOf` the public specification it relates to, and that lists its ontology files (role `schema`) and shapes files (role `validation`) as resources. This is the `dx-prof` vocabulary (`http://www.w3.org/ns/dx/prof/`) that the SHACL 1.2 Profiling specification builds on; see the [SHACL Profiling guide](04e-shacl-profiling.ipynb) for the background.

`get_conformance_profiles()` loads the document as a graph (`url=` loads a different copy instead), and `get_resource_iri(name)` returns the identity an entry has in it.

In [5]:
profiles = starontology.get_conformance_profiles()

rows = profiles.query("""
    PREFIX prof: <http://www.w3.org/ns/dx/prof/>
    SELECT ?profile ?spec ?role ?file WHERE {
      ?profile a prof:Profile ; prof:isProfileOf ?spec ; prof:hasResource ?resource .
      ?resource prof:hasRole ?role ; prof:hasArtifact ?file .
    } ORDER BY ?profile ?role ?file
""")

def short(term):
    return str(term).rsplit("/", 1)[-1].rsplit("#", 1)[-1]

for profile, spec, role, file in rows:
    print(f"{short(profile):12} {short(spec):18} {short(role):11} {short(file)}")

print()
print("the whole document:", len(profiles), "triples")

Manchester   owl2-manchester-syntax schema      manchester-ast-ontology.ttl
Manchester   owl2-manchester-syntax validation  manchester_shapes.ttl
Shacl        rules              validation  shacl12-shacl.ttl
Shacl        node-expr          validation  shacl12-shacl.ttl
Shacl        sparql             validation  shacl12-shacl.ttl
Shacl        core               validation  shacl12-shacl.ttl
Shacl        profiling          validation  shacl12-shacl.ttl
Skos         skos-reference     schema      skos-ontology.ttl
Skos         skos-reference     validation  skos_shapes.ttl
SparqlQuery  sparql12-query     schema      spexpr-ontology.ttl
SparqlQuery  sparql12-query     schema      sqe-ontology.ttl
SparqlQuery  sparql12-query     validation  spexpr_shapes.ttl
SparqlQuery  sparql12-query     validation  sqe_shapes.ttl
SparqlRL     sparql12-rl        schema      srl-ontology.ttl
SparqlRL     sparql12-rl        validation  srl_shapes.ttl

the whole document: 124 triples


## 5. The full conformance chain

`get_resource_iri()` gives a validated dataset's own `shapes_graph_resource=` the *same* identity the profiles document lists the shapes file under. `derive_conforms_to()` then follows the chain: data conforms to the shapes graph (`sh:conformsTo`), so to the StarLayer profile that lists that shapes graph (`slp:Skos`), and through its `prof:isProfileOf` link to the public specification (the SKOS Reference).

In [6]:
from starlayer.shacl import shacl_profiling

result = StarShaclSchema(shacl_graph=skos_shacl).validate(
    data_graph=data,
    meta_shacl=False,
    data_graph_iri="http://example.org/graphs/my-skos-data",
    shapes_graph_resource=starontology.get_resource_iri("skos_shacl"),
)

# conforms_only=True: just the sh:conformsTo / dcterms:conformsTo triples
# themselves - not the report graph merged in too (derive_conforms_to()'s
# own default). Keeps this demo to exactly the claims being derived.
derived = shacl_profiling.derive_conforms_to(
    result.report_graph,
    specification_graph=starontology.get_conformance_profiles(),
    conforms_only=True,
)
for subject, predicate, obj in sorted(derived, key=str):
    print(f"{predicate.n3(derived.namespace_manager):20} {obj}")

dcterms:conformsTo   https://github.com/hidden-graph/starlayer/ns/profile#Skos
dcterms:conformsTo   https://www.w3.org/TR/skos-reference
sh:conformsTo        https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl


## 6. Fetching an entry directly from GitHub, outside Python

`get_resource_iri(name)` (section 5 above) is this registry's own stable *identity* for an entry - a GitHub blob URL. Nice to browse, but a plain HTTP GET against a blob URL returns GitHub's own HTML viewer page, not the file's real bytes. `get_raw_url(name)` is a different thing entirely: a `raw.githubusercontent.com` URL that returns the entry's real content - fetchable by anything, independent of StarLayer being installed at all.


In [7]:
print("identity (get_resource_iri):", starontology.get_resource_iri("skos_shacl"))
print("fetch   (get_raw_url):     ", starontology.get_raw_url("skos_shacl"))


identity (get_resource_iri): https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl
fetch   (get_raw_url):      https://raw.githubusercontent.com/hidden-graph/starlayer/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl


Real content, fetched live over the network - no local file, no bundled package data involved on the reading end:


In [8]:
remote = StarLayerGraph()
remote.parse(starontology.get_raw_url("skos_owl"), format="turtle")
print(len(remote), "triples fetched directly from GitHub")


252 triples fetched directly from GitHub


`get_raw_url()` also covers `"starlayer_profiles"` (the profiles document from section 4) - it has no `get_resource_iri()` counterpart in the registry's own ontology/shapes entries:

In [9]:
print(starontology.get_raw_url("starlayer_profiles"))


https://raw.githubusercontent.com/hidden-graph/starlayer/main/packages/starlayer/starlayer/starontology/starlayer-profiles.ttl
